##### 1. Merge raw PubChem batches
##### Raw PubChem batches are stored in the step 1 folder.
##### The merged dataset is written to the step 2 folder.

In [ ]:
import pandas as pd
import polars as pl
import os
from pathlib import Path
import logging
from datetime import datetime
import gc
import numpy as np
import tempfile
import shutil

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class LargeScaleCSVMerger:
    def __init__(self, input_dir, output_dir=None):
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir) if output_dir else self.input_dir.parent
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.temp_dir = None
        
    def get_csv_files(self):
        """CSV"""
        csv_files = list(self.input_dir.glob("*.csv"))
        logger.info(f" {len(csv_files)} CSV")
        return sorted(csv_files)
    
    def estimate_memory_usage(self, sample_file, total_files):
        """"""
        try:
            # 
            sample_df = pd.read_csv(sample_file, nrows=1000)
            row_memory = sample_df.memory_usage(deep=True).sum() / 1000  # 
            
            # 
            file_size = sample_file.stat().st_size
            sample_size = sample_df.memory_usage(deep=True).sum()
            estimated_rows_per_file = (file_size / sample_size) * 1000
            
            total_estimated_memory = row_memory * estimated_rows_per_file * total_files / (1024**3)  # GB
            logger.info(f": {total_estimated_memory:.2f} GB")
            
            return total_estimated_memory
        except:
            return None
    
    def ultra_low_memory_merge(self, files_per_merge=3):
        """，"""
        logger.info(f"， {files_per_merge} ...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 
        self.temp_dir = Path(tempfile.mkdtemp(dir=self.output_dir))
        
        try:
            # ：CSVParquet
            logger.info("：...")
            parquet_files = []
            
            for i, csv_file in enumerate(csv_files):
                logger.info(f" {i+1}/{len(csv_files)}: {csv_file.name}")
                
                # CSV
                df = pl.read_csv(str(csv_file))
                df = df.with_columns(pl.lit(csv_file.name).alias("source_file"))
                
                # 
                df_unique = df.unique(subset=["SMILES"])
                
                # Parquet
                output_file = self.temp_dir / f"file_{i:04d}.parquet"
                df_unique.write_parquet(str(output_file))
                parquet_files.append(output_file)
                
                logger.info(f"  : {df.shape[0]:,}, : {df_unique.shape[0]:,}")
                del df, df_unique
                gc.collect()
            
            # ：
            logger.info(f"：（{files_per_merge}）...")
            current_files = parquet_files
            generation = 1
            
            while len(current_files) > 1:
                logger.info(f" {generation} ， {len(current_files)} ")
                next_generation = []
                
                # 
                for i in range(0, len(current_files), files_per_merge):
                    group = current_files[i:i+files_per_merge]
                    if len(group) == 1 and i + files_per_merge >= len(current_files):
                        # ，
                        next_generation.append(group[0])
                    else:
                        logger.info(f"   {i//files_per_merge + 1} （{len(group)} ）")
                        
                        # 
                        dfs = [pl.read_parquet(str(f)) for f in group]
                        
                        # 
                        merged = pl.concat(dfs).unique(subset=["SMILES"])
                        
                        # 
                        output_file = self.temp_dir / f"gen_{generation}_group_{i//files_per_merge}.parquet"
                        merged.write_parquet(str(output_file))
                        next_generation.append(output_file)
                        
                        logger.info(f"    : {merged.shape[0]:,} ")
                        
                        # 
                        del dfs, merged
                        gc.collect()
                
                current_files = next_generation
                generation += 1
            
            # CSV
            final_output = self.output_dir / f"merged_data_{timestamp}.csv"
            logger.info("CSV...")
            final_df = pl.read_parquet(str(current_files[0]))
            final_df.write_csv(str(final_output))
            
            logger.info(f"！ {final_df.shape[0]:,} ")
            return final_output
            
        finally:
            if self.temp_dir and self.temp_dir.exists():
                shutil.rmtree(self.temp_dir)
                logger.info("")
    
    def polars_streaming_merge(self, add_source_column=True):
        """Polars"""
        logger.info("Polars...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 
        self.temp_dir = Path(tempfile.mkdtemp(dir=self.output_dir))
        logger.info(f": {self.temp_dir}")
        
        try:
            # ：，
            batch_size = 10  # 10
            batch_outputs = []
            
            for i in range(0, len(csv_files), batch_size):
                batch_files = csv_files[i:i+batch_size]
                batch_num = i // batch_size + 1
                logger.info(f" {batch_num}/{(len(csv_files)-1)//batch_size + 1}")
                
                # lazy evaluation
                lazy_dfs = []
                for csv_file in batch_files:
                    lazy_df = pl.scan_csv(str(csv_file))
                    if add_source_column:
                        lazy_df = lazy_df.with_columns(pl.lit(csv_file.name).alias("source_file"))
                    lazy_dfs.append(lazy_df)
                
                # 
                batch_df = pl.concat(lazy_dfs)
                batch_unique = batch_df.unique(subset=["SMILES"]).collect()
                
                # 
                batch_output = self.temp_dir / f"batch_{batch_num:04d}.parquet"
                batch_unique.write_parquet(str(batch_output))
                batch_outputs.append(batch_output)
                
                logger.info(f" {batch_num} ， {batch_unique.shape[0]:,} ")
                del batch_unique
                gc.collect()
            
            # ：
            logger.info("...")
            final_output = self.output_dir / f"merged_data_{timestamp}.csv"
            
            # 
            current_files = batch_outputs.copy()
            merge_round = 1
            
            while len(current_files) > 1:
                logger.info(f" {merge_round}， {len(current_files)} ")
                next_round_files = []
                
                # 
                for i in range(0, len(current_files), 2):
                    if i + 1 < len(current_files):
                        # 
                        file1, file2 = current_files[i], current_files[i+1]
                        logger.info(f"   {i+1}  {i+2}")
                        
                        # 
                        df1 = pl.read_parquet(str(file1))
                        df2 = pl.read_parquet(str(file2))
                        
                        # 
                        merged = pl.concat([df1, df2]).unique(subset=["SMILES"])
                        
                        # 
                        output_file = self.temp_dir / f"round_{merge_round}_part_{i//2}.parquet"
                        merged.write_parquet(str(output_file))
                        next_round_files.append(output_file)
                        
                        logger.info(f"     {merged.shape[0]:,} ")
                        
                        # 
                        del df1, df2, merged
                        gc.collect()
                    else:
                        # ，
                        next_round_files.append(current_files[i])
                
                current_files = next_round_files
                merge_round += 1
            
            # 
            logger.info("CSV...")
            final_df = pl.read_parquet(str(current_files[0]))
            
            # CSV
            logger.info(f"， {final_df.shape[0]:,} ")
            final_df.write_csv(str(final_output))
            
            return final_output
            
        finally:
            # 
            if self.temp_dir and self.temp_dir.exists():
                shutil.rmtree(self.temp_dir)
                logger.info("")
    
    def chunk_based_dedup(self, chunk_size=1000000):
        """，"""
        logger.info("...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = self.output_dir / f"merged_data_{timestamp}.csv"
        
        # SMILES
        seen_smiles = set()
        total_processed = 0
        total_unique = 0
        first_write = True
        
        # 
        for i, csv_file in enumerate(csv_files):
            logger.info(f" {i+1}/{len(csv_files)}: {csv_file.name}")
            
            # 
            for chunk_num, chunk in enumerate(pd.read_csv(csv_file, chunksize=chunk_size)):
                # 
                chunk['source_file'] = csv_file.name
                
                # SMILES
                mask = ~chunk['SMILES'].isin(seen_smiles)
                unique_chunk = chunk[mask]
                
                # SMILES
                seen_smiles.update(unique_chunk['SMILES'].values)
                
                # 
                if len(unique_chunk) > 0:
                    unique_chunk.to_csv(output_path, mode='w' if first_write else 'a',
                                      header=first_write, index=False)
                    first_write = False
                
                total_processed += len(chunk)
                total_unique += len(unique_chunk)
                
                # 
                if chunk_num % 10 == 0:
                    logger.info(f"   {total_processed:,} ， {total_unique:,} ")
                    logger.info(f"  SMILES: {len(seen_smiles):,}")
            
            # 
            gc.collect()
            
            # SMILES，
            if len(seen_smiles) > 50000000:  # 5000
                logger.warning("SMILES，")
        
        logger.info(f"！ {total_processed:,} ， {total_unique:,} ")
        return output_path

def main():
    # 
    input_directory = r"/home/hank/code/unimol_tools/filter data/step 1"
    output_directory = r"/home/hank/code/unimol_tools/filter data/step 2"
    
    print("CSV")
    print("=" * 50)
    print(f": {input_directory}")
    print(f": {output_directory}")
    print()
    
    # 
    merger = LargeScaleCSVMerger(input_directory, output_directory)
    
    # 
    csv_files = merger.get_csv_files()
    if not csv_files:
        print("CSV")
        return
    
    # 
    print("\n...")
    estimated_memory = merger.estimate_memory_usage(csv_files[0], len(csv_files))
    if estimated_memory and estimated_memory > 16:  # 16GB
        print(f"⚠️  ： {estimated_memory:.1f} GB ，")
    
    # 
    print("\n:")
    print("1. Polars（，）")
    print("2. （，）")
    print("3. （，）")
    print("4. （，）")
    
    choice = input(" (1/2/3/4, 4): ").strip()
    
    try:
        if choice == '2':
            print("\n...")
            output_path = merger.chunk_based_dedup(chunk_size=500000)
        elif choice == '3':
            print("\n...")
            print("：1%，SMILES")
            output_path = merger.bloom_filter_dedup(expected_items=150000000)
        elif choice == '1':
            print("\nPolars...")
            output_path = merger.polars_streaming_merge()
        else:
            print("\n...")
            files_per_merge = input("？(3，): ").strip()
            files_per_merge = int(files_per_merge) if files_per_merge else 3
            output_path = merger.ultra_low_memory_merge(files_per_merge=files_per_merge)
        
        if output_path:
            print(f"\n✅ !")
            print(f"📁 : {output_path}")
            
            # 
            file_size = output_path.stat().st_size / (1024**3)
            print(f"📊 : {file_size:.2f} GB")
        else:
            print("❌ ")
            
    except MemoryError:
        print("\n❌ ！")
        print("：")
        print("1. 2（）")
        print("2. chunk_size")
        print("3. ")
        print("4. ")
    except Exception as e:
        logger.error(f": {e}")
        print(f"❌ : {e}")

if __name__ == "__main__":
    main()
    
    def bloom_filter_dedup(self, expected_items=150000000, false_positive_rate=0.01):
        """，"""
        try:
            from pybloom_live import BloomFilter
        except ImportError:
            logger.error("pybloom-live: pip install pybloom-live")
            return None
        
        logger.info("...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = self.output_dir / f"merged_data_{timestamp}.csv"
        
        # 
        bloom = BloomFilter(capacity=expected_items, error_rate=false_positive_rate)
        
        total_processed = 0
        total_unique = 0
        first_write = True
        chunk_size = 100000
        
        for i, csv_file in enumerate(csv_files):
            logger.info(f" {i+1}/{len(csv_files)}: {csv_file.name}")
            
            for chunk in pd.read_csv(csv_file, chunksize=chunk_size):
                chunk['source_file'] = csv_file.name
                
                # SMILES
                unique_rows = []
                for _, row in chunk.iterrows():
                    smiles = row['SMILES']
                    if smiles not in bloom:
                        bloom.add(smiles)
                        unique_rows.append(row)
                
                # 
                if unique_rows:
                    unique_df = pd.DataFrame(unique_rows)
                    unique_df.to_csv(output_path, mode='w' if first_write else 'a',
                                    header=first_write, index=False)
                    first_write = False
                    total_unique += len(unique_df)
                
                total_processed += len(chunk)
                
                if total_processed % 1000000 == 0:
                    logger.info(f"   {total_processed:,} ， {total_unique:,} ")
            
            gc.collect()
        
        logger.info(f"！ {total_processed:,} ， {total_unique:,} ")
        logger.info(f"：， {false_positive_rate*100}% ")
        return output_path

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen
import os
from pathlib import Path
import logging

# 
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class MoleculeFilter:
    def __init__(self):
        # （）
        self.allowed_elements = {'H', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'Br', 'I'}
        
        # （Table S14）
        self.forbidden_elements = {
            # 
            'He', 'Ne', 'Ar', 'Kr', 'Xe', 'Rn',
            # 
            'Li', 'Na', 'K', 'Rb', 'Cs', 'Fr',
            # d
            'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
            'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
            'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
            'Lr', 'Rf',
            # f
            'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy',
            'Ho', 'Er', 'Tm', 'Yb', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu',
            'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No',
            # 
            'Mg', 'Ca', 'Sr', 'Ba', 'Ra', 'Ga', 'In', 'Tl', 'Ge', 'Sn',
            'Pb', 'As', 'Sb', 'Bi', 'Se', 'Te', 'Po', 'At'
        }
        
        # SMARTS
        self.forbidden_patterns = [
            # 
            '[OH]',  # 
            '[NH2]',  # 
            '[NH][#6]',  # 
            '[SH]',  # 
            # 
            'C=C',  # 
            'C#C',  # 
            'C=N',  # 
            'N=C',  # 
            # （）
            'c',  # 
            #  - 
            '[*+]',  # 
            '[*-]',  # 
            '[*+2]', # +2
            '[*-2]', # -2
            '[*+3]', # +3
            '[*-3]', # -3
        ]

    def is_valid_molecular_weight(self, mol_weight):
        """"""
        return 0 <= mol_weight <= 600

    def is_valid_heavy_atom_count(self, heavy_atom_count):
        """"""
        return 0 <= heavy_atom_count <= 30

    def contains_allowed_elements_only(self, smiles):
        """"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return False
            
            for atom in mol.GetAtoms():
                element = atom.GetSymbol()
                if element not in self.allowed_elements:
                    return False
                if element in self.forbidden_elements:
                    return False
            return True
        except:
            return False

    def has_charged_atoms(self, smiles):
        """（）"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return True
            
            # 
            for atom in mol.GetAtoms():
                if atom.GetFormalCharge() != 0:
                    return True
            
            # molecules
            total_charge = Chem.rdmolops.GetFormalCharge(mol)
            if total_charge != 0:
                return True
                
            return False
        except:
            return True

    def contains_forbidden_patterns(self, smiles):
        """"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return True
            
            # （）
            if self.has_charged_atoms(smiles):
                return True
            
            # 
            for pattern in self.forbidden_patterns:
                # ，has_charged_atoms
                if '*+' in pattern or '*-' in pattern:
                    continue
                if mol.HasSubstructMatch(Chem.MolFromSmarts(pattern)):
                    return True
            return False
        except:
            return True

    def is_aromatic(self, smiles):
        """"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return True
            
            # 
            for atom in mol.GetAtoms():
                if atom.GetIsAromatic():
                    return True
            return False
        except:
            return True

    def has_isotopes(self, smiles):
        """"""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return True
            
            for atom in mol.GetAtoms():
                if atom.GetIsotope() != 0:
                    return True
            return False
        except:
            return True

    def has_stereo_centers(self, smiles):
        """（）"""
        try:
            # SMILES
            if '@' in smiles or '@@' in smiles:
                return True
            return False
        except:
            return True

    def filter_molecule(self, row):
        """molecules"""
        try:
            smiles = row['SMILES']
            mol_weight = row['Molecular_Weight']
            heavy_atom_count = row['Heavy_Atom_Count']
            
            # 
            if not self.is_valid_molecular_weight(mol_weight):
                return False, ""
            
            if not self.is_valid_heavy_atom_count(heavy_atom_count):
                return False, ""
            
            # 
            if not self.contains_allowed_elements_only(smiles):
                return False, ""
            
            # （）
            if self.has_charged_atoms(smiles):
                return False, ""
            
            # 
            if self.contains_forbidden_patterns(smiles):
                return False, ""
            
            # 
            if self.is_aromatic(smiles):
                return False, ""
            
            # 
            if self.has_isotopes(smiles):
                return False, ""
            
            # 
            if self.has_stereo_centers(smiles):
                return False, ""
            
            return True, ""
            
        except Exception as e:
            return False, f": {str(e)}"

def test_charged_molecules():
    """"""
    test_cases = [
        ("F[Si-2](F)(F)(F)(F)F", ""),
        ("CC(C)(C)[N+](C)(C)C", ""),
        ("CC(=O)[O-]", ""),
        ("CCO", "（）"),
        ("[Na+].[Cl-]", ""),
        ("CCCC", "（）"),
    ]
    
    filter_obj = MoleculeFilter()
    print("：")
    print("-" * 50)
    
    for smiles, description in test_cases:
        has_charge = filter_obj.has_charged_atoms(smiles)
        print(f"SMILES: {smiles:<30} | {description:<15} | : {has_charge}")

def main():
    # 
    test_charged_molecules()
    print("\n")
    
    # 
    input_file = r"/home/hank/code/unimol_tools/filter data/step 2/merged_data_20250619_143443.csv"
    output_dir = r"/home/hank/code/unimol_tools/filter data/step 3"
    
    # 
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # 
    input_filename = Path(input_file).stem
    output_file = os.path.join(output_dir, f"{input_filename}_filtered.csv")
    report_file = os.path.join(output_dir, f"{input_filename}_filter_report.txt")
    
    logger.info(f": {input_file}")
    
    try:
        # 
        df = pd.read_csv(input_file)
        logger.info(f" {len(df)} ")
        
        # 
        required_columns = ['SMILES', 'Molecular_Weight', 'Heavy_Atom_Count']
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if missing_columns:
            logger.error(f": {missing_columns}")
            logger.info(f": {list(df.columns)}")
            return
        
        # 
        filter_obj = MoleculeFilter()
        
        # 
        filtered_data = []
        filter_reasons = {}
        
        logger.info("...")
        
        # 
        for idx, row in df.iterrows():
            if idx % 1000 == 0:
                logger.info(f" {idx}/{len(df)} ")
            
            is_valid, reason = filter_obj.filter_molecule(row)
            
            if is_valid:
                filtered_data.append(row)
            else:
                filter_reasons[reason] = filter_reasons.get(reason, 0) + 1
        
        # DataFrame
        filtered_df = pd.DataFrame(filtered_data)
        
        # 
        filtered_df.to_csv(output_file, index=False)
        logger.info(f"！: {output_file}")
        
        # 
        with open(report_file, 'w', encoding='utf-8') as f:
            f.write("PubChem\n")
            f.write("=" * 50 + "\n\n")
            f.write(f": {input_file}\n")
            f.write(f": {output_file}\n\n")
            f.write(f": {len(df)}\n")
            f.write(f": {len(filtered_df)}\n")
            f.write(f": {len(filtered_df)/len(df)*100:.2f}%\n\n")
            f.write(":\n")
            f.write("- : 0-600 g/mol\n")
            f.write("- : 0-30\n")
            f.write("- : H, C, N, O, F, Si, P, S, Cl, Br, I\n")
            f.write("- : 、、、、、\n\n")
            f.write(":\n")
            for reason, count in sorted(filter_reasons.items(), key=lambda x: x[1], reverse=True):
                f.write(f"- {reason}: {count} \n")
        
        logger.info(f": {report_file}")
        logger.info(f"！: {len(df)} → : {len(filtered_df)} (: {len(filtered_df)/len(df)*100:.2f}%)")
        
        # 
        print("\n:")
        print(f": {len(df)} ")
        print(f": {len(filtered_df)} ")
        print(f": {len(filtered_df)/len(df)*100:.2f}%")
        print("\n:")
        for reason, count in sorted(filter_reasons.items(), key=lambda x: x[1], reverse=True):
            print(f"- {reason}: {count} ")
        
    except Exception as e:
        logger.error(f": {str(e)}")
        raise

if __name__ == "__main__":
    main()

##### 3. step 3
#####   step 4

In [ ]:
import pandas as pd
import polars as pl
import os
from pathlib import Path
import logging
from datetime import datetime
import gc
import numpy as np
import tempfile
import shutil

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class LargeScaleCSVMerger:
    def __init__(self, input_dir, output_dir=None):
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir) if output_dir else self.input_dir.parent
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.temp_dir = None
        
    def get_csv_files(self):
        """CSV"""
        csv_files = list(self.input_dir.glob("*.csv"))
        logger.info(f" {len(csv_files)} CSV")
        return sorted(csv_files)
    
    def estimate_memory_usage(self, sample_file, total_files):
        """"""
        try:
            # 
            sample_df = pd.read_csv(sample_file, nrows=1000)
            row_memory = sample_df.memory_usage(deep=True).sum() / 1000  # 
            
            # 
            file_size = sample_file.stat().st_size
            sample_size = sample_df.memory_usage(deep=True).sum()
            estimated_rows_per_file = (file_size / sample_size) * 1000
            
            total_estimated_memory = row_memory * estimated_rows_per_file * total_files / (1024**3)  # GB
            logger.info(f": {total_estimated_memory:.2f} GB")
            
            return total_estimated_memory
        except:
            return None
    
    def ultra_low_memory_merge(self, files_per_merge=3):
        """，"""
        logger.info(f"， {files_per_merge} ...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 
        self.temp_dir = Path(tempfile.mkdtemp(dir=self.output_dir))
        
        try:
            # ：CSVParquet
            logger.info("：...")
            parquet_files = []
            
            for i, csv_file in enumerate(csv_files):
                logger.info(f" {i+1}/{len(csv_files)}: {csv_file.name}")
                
                # CSV
                df = pl.read_csv(str(csv_file))
                df = df.with_columns(pl.lit(csv_file.name).alias("source_file"))
                
                # 
                df_unique = df.unique(subset=["SMILES"])
                
                # Parquet
                output_file = self.temp_dir / f"file_{i:04d}.parquet"
                df_unique.write_parquet(str(output_file))
                parquet_files.append(output_file)
                
                logger.info(f"  : {df.shape[0]:,}, : {df_unique.shape[0]:,}")
                del df, df_unique
                gc.collect()
            
            # ：
            logger.info(f"：（{files_per_merge}）...")
            current_files = parquet_files
            generation = 1
            
            while len(current_files) > 1:
                logger.info(f" {generation} ， {len(current_files)} ")
                next_generation = []
                
                # 
                for i in range(0, len(current_files), files_per_merge):
                    group = current_files[i:i+files_per_merge]
                    if len(group) == 1 and i + files_per_merge >= len(current_files):
                        # ，
                        next_generation.append(group[0])
                    else:
                        logger.info(f"   {i//files_per_merge + 1} （{len(group)} ）")
                        
                        # 
                        dfs = [pl.read_parquet(str(f)) for f in group]
                        
                        # 
                        merged = pl.concat(dfs).unique(subset=["SMILES"])
                        
                        # 
                        output_file = self.temp_dir / f"gen_{generation}_group_{i//files_per_merge}.parquet"
                        merged.write_parquet(str(output_file))
                        next_generation.append(output_file)
                        
                        logger.info(f"    : {merged.shape[0]:,} ")
                        
                        # 
                        del dfs, merged
                        gc.collect()
                
                current_files = next_generation
                generation += 1
            
            # CSV
            final_output = self.output_dir / f"merged_data_{timestamp}.csv"
            logger.info("CSV...")
            final_df = pl.read_parquet(str(current_files[0]))
            final_df.write_csv(str(final_output))
            
            logger.info(f"！ {final_df.shape[0]:,} ")
            return final_output
            
        finally:
            if self.temp_dir and self.temp_dir.exists():
                shutil.rmtree(self.temp_dir)
                logger.info("")
    
    def polars_streaming_merge(self, add_source_column=True):
        """Polars"""
        logger.info("Polars...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # 
        self.temp_dir = Path(tempfile.mkdtemp(dir=self.output_dir))
        logger.info(f": {self.temp_dir}")
        
        try:
            # ：，
            batch_size = 10  # 10
            batch_outputs = []
            
            for i in range(0, len(csv_files), batch_size):
                batch_files = csv_files[i:i+batch_size]
                batch_num = i // batch_size + 1
                logger.info(f" {batch_num}/{(len(csv_files)-1)//batch_size + 1}")
                
                # lazy evaluation
                lazy_dfs = []
                for csv_file in batch_files:
                    lazy_df = pl.scan_csv(str(csv_file))
                    if add_source_column:
                        lazy_df = lazy_df.with_columns(pl.lit(csv_file.name).alias("source_file"))
                    lazy_dfs.append(lazy_df)
                
                # 
                batch_df = pl.concat(lazy_dfs)
                batch_unique = batch_df.unique(subset=["SMILES"]).collect()
                
                # 
                batch_output = self.temp_dir / f"batch_{batch_num:04d}.parquet"
                batch_unique.write_parquet(str(batch_output))
                batch_outputs.append(batch_output)
                
                logger.info(f" {batch_num} ， {batch_unique.shape[0]:,} ")
                del batch_unique
                gc.collect()
            
            # ：
            logger.info("...")
            final_output = self.output_dir / f"merged_data_{timestamp}.csv"
            
            # 
            current_files = batch_outputs.copy()
            merge_round = 1
            
            while len(current_files) > 1:
                logger.info(f" {merge_round}， {len(current_files)} ")
                next_round_files = []
                
                # 
                for i in range(0, len(current_files), 2):
                    if i + 1 < len(current_files):
                        # 
                        file1, file2 = current_files[i], current_files[i+1]
                        logger.info(f"   {i+1}  {i+2}")
                        
                        # 
                        df1 = pl.read_parquet(str(file1))
                        df2 = pl.read_parquet(str(file2))
                        
                        # 
                        merged = pl.concat([df1, df2]).unique(subset=["SMILES"])
                        
                        # 
                        output_file = self.temp_dir / f"round_{merge_round}_part_{i//2}.parquet"
                        merged.write_parquet(str(output_file))
                        next_round_files.append(output_file)
                        
                        logger.info(f"     {merged.shape[0]:,} ")
                        
                        # 
                        del df1, df2, merged
                        gc.collect()
                    else:
                        # ，
                        next_round_files.append(current_files[i])
                
                current_files = next_round_files
                merge_round += 1
            
            # 
            logger.info("CSV...")
            final_df = pl.read_parquet(str(current_files[0]))
            
            # CSV
            logger.info(f"， {final_df.shape[0]:,} ")
            final_df.write_csv(str(final_output))
            
            return final_output
            
        finally:
            # 
            if self.temp_dir and self.temp_dir.exists():
                shutil.rmtree(self.temp_dir)
                logger.info("")
    
    def chunk_based_dedup(self, chunk_size=1000000):
        """，"""
        logger.info("...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = self.output_dir / f"merged_data_{timestamp}.csv"
        
        # SMILES
        seen_smiles = set()
        total_processed = 0
        total_unique = 0
        first_write = True
        
        # 
        for i, csv_file in enumerate(csv_files):
            logger.info(f" {i+1}/{len(csv_files)}: {csv_file.name}")
            
            # 
            for chunk_num, chunk in enumerate(pd.read_csv(csv_file, chunksize=chunk_size)):
                # 
                chunk['source_file'] = csv_file.name
                
                # SMILES
                mask = ~chunk['SMILES'].isin(seen_smiles)
                unique_chunk = chunk[mask]
                
                # SMILES
                seen_smiles.update(unique_chunk['SMILES'].values)
                
                # 
                if len(unique_chunk) > 0:
                    unique_chunk.to_csv(output_path, mode='w' if first_write else 'a',
                                      header=first_write, index=False)
                    first_write = False
                
                total_processed += len(chunk)
                total_unique += len(unique_chunk)
                
                # 
                if chunk_num % 10 == 0:
                    logger.info(f"   {total_processed:,} ， {total_unique:,} ")
                    logger.info(f"  SMILES: {len(seen_smiles):,}")
            
            # 
            gc.collect()
            
            # SMILES，
            if len(seen_smiles) > 50000000:  # 5000
                logger.warning("SMILES，")
        
        logger.info(f"！ {total_processed:,} ， {total_unique:,} ")
        return output_path

def main():
    # 
    input_directory = r"/home/hank/code/unimol_tools/filter data/step 3"
    output_directory = r"/home/hank/code/unimol_tools/filter data/step 4"
    
    print("CSV")
    print("=" * 50)
    print(f": {input_directory}")
    print(f": {output_directory}")
    print()
    
    # 
    merger = LargeScaleCSVMerger(input_directory, output_directory)
    
    # 
    csv_files = merger.get_csv_files()
    if not csv_files:
        print("CSV")
        return
    
    # 
    print("\n...")
    estimated_memory = merger.estimate_memory_usage(csv_files[0], len(csv_files))
    if estimated_memory and estimated_memory > 16:  # 16GB
        print(f"⚠️  ： {estimated_memory:.1f} GB ，")
    
    # 
    print("\n:")
    print("1. Polars（，）")
    print("2. （，）")
    print("3. （，）")
    print("4. （，）")
    
    choice = input(" (1/2/3/4, 4): ").strip()
    
    try:
        if choice == '2':
            print("\n...")
            output_path = merger.chunk_based_dedup(chunk_size=500000)
        elif choice == '3':
            print("\n...")
            print("：1%，SMILES")
            output_path = merger.bloom_filter_dedup(expected_items=150000000)
        elif choice == '1':
            print("\nPolars...")
            output_path = merger.polars_streaming_merge()
        else:
            print("\n...")
            files_per_merge = input("？(3，): ").strip()
            files_per_merge = int(files_per_merge) if files_per_merge else 3
            output_path = merger.ultra_low_memory_merge(files_per_merge=files_per_merge)
        
        if output_path:
            print(f"\n✅ !")
            print(f"📁 : {output_path}")
            
            # 
            file_size = output_path.stat().st_size / (1024**3)
            print(f"📊 : {file_size:.2f} GB")
        else:
            print("❌ ")
            
    except MemoryError:
        print("\n❌ ！")
        print("：")
        print("1. 2（）")
        print("2. chunk_size")
        print("3. ")
        print("4. ")
    except Exception as e:
        logger.error(f": {e}")
        print(f"❌ : {e}")

if __name__ == "__main__":
    main()
    
    def bloom_filter_dedup(self, expected_items=150000000, false_positive_rate=0.01):
        """，"""
        try:
            from pybloom_live import BloomFilter
        except ImportError:
            logger.error("pybloom-live: pip install pybloom-live")
            return None
        
        logger.info("...")
        
        csv_files = self.get_csv_files()
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = self.output_dir / f"merged_data_{timestamp}.csv"
        
        # 
        bloom = BloomFilter(capacity=expected_items, error_rate=false_positive_rate)
        
        total_processed = 0
        total_unique = 0
        first_write = True
        chunk_size = 100000
        
        for i, csv_file in enumerate(csv_files):
            logger.info(f" {i+1}/{len(csv_files)}: {csv_file.name}")
            
            for chunk in pd.read_csv(csv_file, chunksize=chunk_size):
                chunk['source_file'] = csv_file.name
                
                # SMILES
                unique_rows = []
                for _, row in chunk.iterrows():
                    smiles = row['SMILES']
                    if smiles not in bloom:
                        bloom.add(smiles)
                        unique_rows.append(row)
                
                # 
                if unique_rows:
                    unique_df = pd.DataFrame(unique_rows)
                    unique_df.to_csv(output_path, mode='w' if first_write else 'a',
                                    header=first_write, index=False)
                    first_write = False
                    total_unique += len(unique_df)
                
                total_processed += len(chunk)
                
                if total_processed % 1000000 == 0:
                    logger.info(f"   {total_processed:,} ， {total_unique:,} ")
            
            gc.collect()
        
        logger.info(f"！ {total_processed:,} ， {total_unique:,} ")
        logger.info(f"：， {false_positive_rate*100}% ")
        return output_path

##### 4. step 4 
#####   step 5 

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
UniMol - UniMol
unimol_toolsSMILES
"""

import os
import sys
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# unimol_tools
try:
    from unimol_tools.data.conformer import mol2unimolv2, UniMolV2Feature
    USE_ACTUAL_UNIMOL = True
    print("UniMol")
except ImportError:
    USE_ACTUAL_UNIMOL = False
    print("UniMol，")

def test_smiles_with_actual_unimol(smiles, max_atoms=256, remove_hs=True):
    """
    UniMolSMILES
    : (is_valid, error_type, error_detail)
    """
    try:
        # 1: SMILES
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False, "invalid_smiles", "Cannot parse SMILES"
        
        # 2: mol2unimolv2 - 
        try:
            result = mol2unimolv2(mol, max_atoms=max_atoms, remove_hs=remove_hs)
            if result is None:
                return False, "mol2unimolv2_failed", "mol2unimolv2 returned None"
            return True, "valid", "OK"
        except Chem.rdchem.KekulizeException as e:
            # 
            return False, "kekulize_error", str(e)
        except Exception as e:
            return False, "mol2unimolv2_error", str(e)
            
    except Exception as e:
        return False, "unexpected_error", str(e)

def test_smiles_simulation(smiles, max_atoms=256):
    """
    UniMol（）
    """
    try:
        # 
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False, "invalid_smiles", "Cannot parse SMILES"
        
        # ：RemoveAllHs - 
        try:
            # 
            mol_with_h = Chem.AddHs(mol)
            #  - KekulizeException
            mol_no_h = AllChem.RemoveAllHs(mol_with_h)
        except Chem.rdchem.KekulizeException as e:
            return False, "kekulize_error", str(e)
        except Exception as e:
            return False, "remove_h_error", str(e)
        
        if mol_no_h is None or mol_no_h.GetNumAtoms() == 0:
            return False, "no_atoms", "No atoms after removing H"
        
        # 
        if mol_no_h.GetNumAtoms() > max_atoms:
            return False, "too_many_atoms", f"Molecule has {mol_no_h.GetNumAtoms()} atoms, max is {max_atoms}"
        
        return True, "valid", "OK"
        
    except Exception as e:
        return False, "unexpected_error", str(e)

def test_single_process_method(smiles):
    """
    UniMolV2Featuresingle_process
    
    """
    if not USE_ACTUAL_UNIMOL:
        return test_smiles_simulation(smiles)
    
    try:
        # UniMolV2Feature
        feature_extractor = UniMolV2Feature(
            max_atoms=256,
            remove_hs=True,
            multi_process=False  # 
        )
        
        # single_process
        result = feature_extractor.single_process(smiles)
        if result is None:
            return False, "single_process_none", "single_process returned None"
        
        return True, "valid", "OK"
        
    except Chem.rdchem.KekulizeException as e:
        return False, "kekulize_error", str(e)
    except Exception as e:
        return False, "single_process_error", str(e)

def clean_smiles_file(input_file, output_file, test_method="actual", 
                     log_file="cleaning_log.txt", error_details_file="error_details.csv"):
    """
    SMILES
    
    Args:
        input_file: 
        output_file: 
        test_method:  - "actual"UniMol, "simulate", "single_process"single_process
        log_file: 
        error_details_file: 
    """
    print(f": {input_file}")
    print(f": {test_method}")
    
    # 
    df = pd.read_csv(input_file)
    
    # SMILES
    if 'smiles' in df.columns:
        smiles_col = 'smiles'
    elif 'SMILES' in df.columns:
        smiles_col = 'SMILES'
    else:
        smiles_col = df.columns[0]
        print(f" '{smiles_col}' SMILES")
    
    smiles_list = df[smiles_col].tolist()
    
    # 
    if test_method == "single_process" and USE_ACTUAL_UNIMOL:
        test_func = test_single_process_method
    elif test_method == "actual" and USE_ACTUAL_UNIMOL:
        test_func = test_smiles_with_actual_unimol
    else:
        test_func = test_smiles_simulation
        if test_method != "simulate":
            print("：UniMol，")
    
    # SMILES
    valid_indices = []
    error_stats = {}
    error_details = []
    
    print(f" {len(smiles_list)} SMILES...")
    
    for i, smiles in enumerate(tqdm(smiles_list, desc="SMILES")):
        is_valid, error_type, error_detail = test_func(smiles)
        
        if is_valid:
            valid_indices.append(i)
        else:
            error_stats[error_type] = error_stats.get(error_type, 0) + 1
            error_details.append({
                'index': i,
                'smiles': smiles,
                'error_type': error_type,
                'error_detail': error_detail
            })
            
            # Kekulize，
            if error_type == "kekulize_error" and len(error_details) <= 5:
                print(f"\nKekulize #{len([e for e in error_details if e['error_type'] == 'kekulize_error'])}:")
                print(f"  : {i}")
                print(f"  SMILES: {smiles}")
                print(f"  : {error_detail}")
    
    # SMILES
    print(f"\n! SMILES: {len(valid_indices)}/{len(smiles_list)}")
    
    df_valid = df.iloc[valid_indices]
    df_valid.to_csv(output_file, index=False)
    
    # 
    if error_details:
        pd.DataFrame(error_details).to_csv(error_details_file, index=False)
        print(f": {error_details_file}")
    
    # 
    with open(log_file, 'w') as f:
        f.write(f"UniMol\n")
        f.write(f"======================\n")
        f.write(f": {test_method}\n")
        f.write(f": {input_file}\n")
        f.write(f": {output_file}\n")
        f.write(f"SMILES: {len(smiles_list)}\n")
        f.write(f"SMILES: {len(valid_indices)}\n")
        f.write(f"SMILES: {len(smiles_list) - len(valid_indices)}\n")
        f.write(f": {len(valid_indices)/len(smiles_list)*100:.2f}%\n\n")
        
        f.write(f":\n")
        for error_type, count in sorted(error_stats.items(), key=lambda x: x[1], reverse=True):
            f.write(f"  {error_type}: {count}\n")
    
    print(f"\n:")
    for error_type, count in sorted(error_stats.items(), key=lambda x: x[1], reverse=True):
        print(f"  {error_type}: {count}")
    
    print(f"\n!")
    print(f"- : {output_file}")
    print(f"- : {log_file}")
    print(f"- : {error_details_file}")
    
    return df_valid, error_details

def find_kekulize_error_in_file(file_path):
    """
    KekulizeSMILES
    """
    print(f"Kekulize: {file_path}")
    
    df = pd.read_csv(file_path)
    if 'smiles' in df.columns:
        smiles_col = 'smiles'
    elif 'SMILES' in df.columns:
        smiles_col = 'SMILES'
    else:
        smiles_col = df.columns[0]
    
    kekulize_errors = []
    
    for i, smiles in enumerate(df[smiles_col]):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                mol_with_h = Chem.AddHs(mol)
                #  - 
                mol_no_h = AllChem.RemoveAllHs(mol_with_h)
        except Chem.rdchem.KekulizeException as e:
            kekulize_errors.append({
                'index': i,
                'smiles': smiles,
                'error': str(e)
            })
            if len(kekulize_errors) <= 3:
                print(f"\nKekulize #{len(kekulize_errors)}:")
                print(f"  : {i}")
                print(f"  SMILES: {smiles}")
                print(f"  : {e}")
    
    print(f"\n {len(kekulize_errors)} Kekulize")
    return kekulize_errors

if __name__ == "__main__":
    # 
    input_file = r"/home/hank/code/unimol_tools/filter data/step 5/final_cleaned_data.csv"
    output_file = r"/home/hank/code/unimol_tools/filter data/step 5/final_cleaned_data_fixed.csv"
    
    # Kekulize
    print("=== 1: Kekulize ===")
    kekulize_errors = find_kekulize_error_in_file(input_file)
    
    if kekulize_errors:
        print("\nKekulize！。")
        
        # 
        print("\n=== 2:  ===")
        
        # 
        test_methods = ["single_process", "actual", "simulate"]
        
        for method in test_methods:
            print(f"\n: {method}")
            try:
                df_valid, error_details = clean_smiles_file(
                    input_file=input_file,
                    output_file=output_file,
                    test_method=method,
                    log_file=f"cleaning_log_{method}.txt",
                    error_details_file=f"error_details_{method}.csv"
                )
                break
            except Exception as e:
                print(f" {method} : {e}")
                continue
    else:
        print("\nKekulize，。")
        print("，。")

#### 6. unimolv2 clearn

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
UniMolV2
"""

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import AllChem
from unimol_tools.data.conformer import mol2unimolv2, inner_smi2coords

def test_molecule_unimolv2_direct(smiles, max_atoms=256):
    """UniMolV2"""
    try:
        # 1: SMILES
        mol = inner_smi2coords(
            smiles,
            seed=42,
            mode='fast',
            remove_hs=True,
            return_mol=True,
        )
        
        if mol is None:
            return False, "mol_creation_failed", "Failed to create molecule"
        
        # 2: mol2unimolv2 - RemoveAllHs
        try:
            feat = mol2unimolv2(mol, max_atoms, remove_hs=True)
            return True, None, None
        except Exception as e:
            error_str = str(e)
            if "Can't kekulize mol" in error_str:
                return False, "kekulize_error", error_str
            else:
                return False, "mol2unimolv2_error", error_str
                
    except Exception as e:
        return False, "unexpected_error", str(e)

def test_batch_direct(smiles_list, max_atoms=256):
    """SMILES"""
    problematic = []
    for i, smiles in enumerate(smiles_list):
        is_valid, error_type, error_msg = test_molecule_unimolv2_direct(smiles, max_atoms)
        if not is_valid and error_type == "kekulize_error":
            problematic.append(i)
    return problematic

def find_problematic_with_binary_search(smiles_list, max_atoms=256):
    """"""
    if len(smiles_list) <= 10:
        # 
        return test_batch_direct(smiles_list, max_atoms)
    
    # 
    try:
        # 
        for smiles in smiles_list:
            mol = inner_smi2coords(smiles, seed=42, mode='fast', remove_hs=True, return_mol=True)
            if mol:
                feat = mol2unimolv2(mol, max_atoms, remove_hs=True)
        return []  # 
    except Exception as e:
        if "Can't kekulize mol" not in str(e):
            print(f"Kekulize: {e}")
            return []
    
    # 
    mid = len(smiles_list) // 2
    left_problematic = find_problematic_with_binary_search(smiles_list[:mid], max_atoms)
    right_problematic = find_problematic_with_binary_search(smiles_list[mid:], max_atoms)
    
    # 
    right_problematic = [idx + mid for idx in right_problematic]
    
    return left_problematic + right_problematic

def main():
    # 
    input_file = r'/home/hank/code/unimol_tools/filter data/step 5/final_cleaned_data.csv'
    output_file = r'/home/hank/code/unimol_tools/filter data/step 5/final_cleaned_data_kekulize_fixed.csv'
    problematic_molecules_file = r'/home/hank/code/unimol_tools/filter data/step 5/problematic_molecules.csv'
    
    # 
    failed_batches = [
        {'batch_idx': 97, 'start_idx': 96000, 'end_idx': 97000},
        {'batch_idx': 107, 'start_idx': 106000, 'end_idx': 107000},
        {'batch_idx': 159, 'start_idx': 158000, 'end_idx': 159000},
        {'batch_idx': 317, 'start_idx': 316000, 'end_idx': 317000},
        {'batch_idx': 354, 'start_idx': 353000, 'end_idx': 354000},
        {'batch_idx': 501, 'start_idx': 500000, 'end_idx': 501000},
        {'batch_idx': 510, 'start_idx': 509000, 'end_idx': 510000},
        {'batch_idx': 578, 'start_idx': 577000, 'end_idx': 578000},
        {'batch_idx': 617, 'start_idx': 616000, 'end_idx': 617000},
        {'batch_idx': 840, 'start_idx': 839000, 'end_idx': 840000},
        {'batch_idx': 943, 'start_idx': 942000, 'end_idx': 943000},
    ]
    
    print("=== UniMolV2 ===")
    
    # 
    print(f": {input_file}")
    df = pd.read_csv(input_file)
    
    # SMILES
    if 'SMILES' in df.columns:
        smiles_col = 'SMILES'
    elif 'smiles' in df.columns:
        smiles_col = 'smiles'
    else:
        smiles_col = df.columns[0]
    
    problematic_molecules = []
    
    # 
    for batch_info in failed_batches:
        batch_idx = batch_info['batch_idx']
        start_idx = batch_info['start_idx']
        end_idx = batch_info['end_idx']
        
        print(f"\n {batch_idx} ( {start_idx}-{end_idx-1})")
        
        # 
        batch_smiles = df.iloc[start_idx:end_idx][smiles_col].tolist()
        
        # 1: （）
        print(f"  ...")
        found_count = 0
        for i in tqdm(range(len(batch_smiles)), desc=f" {batch_idx}"):
            smiles = batch_smiles[i]
            is_valid, error_type, error_msg = test_molecule_unimolv2_direct(smiles)
            
            if not is_valid and error_type == "kekulize_error":
                global_idx = start_idx + i
                problematic_molecules.append({
                    'index': global_idx,
                    'batch_idx': batch_idx,
                    'smiles': smiles,
                    'error': error_msg
                })
                found_count += 1
                if found_count <= 3:  # 3
                    print(f"    :  {global_idx}")
                    print(f"    SMILES: {smiles}")
                    print(f"    : {error_msg}")
        
        if found_count == 0:
            print(f"  ： {batch_idx} ")
        else:
            print(f"   {batch_idx}  {found_count} ")
    
    if problematic_molecules:
        # 
        pd.DataFrame(problematic_molecules).to_csv(problematic_molecules_file, index=False)
        print(f"\n {len(problematic_molecules)} ")
        print(f": {problematic_molecules_file}")
        
        # 
        print("\n...")
        problematic_indices = [mol['index'] for mol in problematic_molecules]
        
        df_clean = df.drop(index=problematic_indices)
        df_clean.reset_index(drop=True, inplace=True)
        
        df_clean.to_csv(output_file, index=False)
        print(f": {output_file}")
        print(f": {len(df)}")
        print(f": {len(problematic_indices)}")
        print(f": {len(df_clean)}")
        
        # 
        print("\n:")
        batch_stats = {}
        for mol in problematic_molecules:
            batch_idx = mol['batch_idx']
            batch_stats[batch_idx] = batch_stats.get(batch_idx, 0) + 1
        
        for batch_idx in sorted(batch_stats.keys()):
            print(f"   {batch_idx}: {batch_stats[batch_idx]} ")
        
    else:
        print("\n！")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
UniMolV2
"""

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import AllChem
from unimol_tools.data.conformer import mol2unimolv2, inner_smi2coords

def test_molecule_unimolv2_direct(smiles, max_atoms=256):
    """UniMolV2"""
    try:
        # 1: SMILES
        mol = inner_smi2coords(
            smiles,
            seed=42,
            mode='fast',
            remove_hs=True,
            return_mol=True,
        )
        
        if mol is None:
            return False, "mol_creation_failed", "Failed to create molecule"
        
        # 2: mol2unimolv2 - RemoveAllHs
        try:
            feat = mol2unimolv2(mol, max_atoms, remove_hs=True)
            return True, None, None
        except Exception as e:
            error_str = str(e)
            if "Can't kekulize mol" in error_str:
                return False, "kekulize_error", error_str
            else:
                return False, "mol2unimolv2_error", error_str
                
    except Exception as e:
        return False, "unexpected_error", str(e)

def test_batch_direct(smiles_list, max_atoms=256):
    """SMILES"""
    problematic = []
    for i, smiles in enumerate(smiles_list):
        is_valid, error_type, error_msg = test_molecule_unimolv2_direct(smiles, max_atoms)
        if not is_valid and error_type == "kekulize_error":
            problematic.append(i)
    return problematic

def find_problematic_with_binary_search(smiles_list, max_atoms=256):
    """"""
    if len(smiles_list) <= 10:
        # 
        return test_batch_direct(smiles_list, max_atoms)
    
    # 
    try:
        # 
        for smiles in smiles_list:
            mol = inner_smi2coords(smiles, seed=42, mode='fast', remove_hs=True, return_mol=True)
            if mol:
                feat = mol2unimolv2(mol, max_atoms, remove_hs=True)
        return []  # 
    except Exception as e:
        if "Can't kekulize mol" not in str(e):
            print(f"Kekulize: {e}")
            return []
    
    # 
    mid = len(smiles_list) // 2
    left_problematic = find_problematic_with_binary_search(smiles_list[:mid], max_atoms)
    right_problematic = find_problematic_with_binary_search(smiles_list[mid:], max_atoms)
    
    # 
    right_problematic = [idx + mid for idx in right_problematic]
    
    return left_problematic + right_problematic

def main():
    # 
    input_file = r'/home/hank/code/unimol_tools/filter data/step 5/final_cleaned_data.csv'
    output_file = r'/home/hank/code/unimol_tools/filter data/step 5/final_cleaned_data_kekulize_fixed.csv'
    problematic_molecules_file = r'/home/hank/code/unimol_tools/filter data/step 5/problematic_molecules.csv'
    cleaned_batches_file = r'/home/hank/code/unimol_tools/filter data/step 5/cleaned_batches_combined.csv'  # ：
    
    # 
    failed_batches = [
        {'batch_idx': 97, 'start_idx': 96000, 'end_idx': 97000},
        {'batch_idx': 107, 'start_idx': 106000, 'end_idx': 107000},
        {'batch_idx': 159, 'start_idx': 158000, 'end_idx': 159000},
        {'batch_idx': 317, 'start_idx': 316000, 'end_idx': 317000},
        {'batch_idx': 354, 'start_idx': 353000, 'end_idx': 354000},
        {'batch_idx': 501, 'start_idx': 500000, 'end_idx': 501000},
        {'batch_idx': 510, 'start_idx': 509000, 'end_idx': 510000},
        {'batch_idx': 578, 'start_idx': 577000, 'end_idx': 578000},
        {'batch_idx': 617, 'start_idx': 616000, 'end_idx': 617000},
        {'batch_idx': 840, 'start_idx': 839000, 'end_idx': 840000},
        {'batch_idx': 943, 'start_idx': 942000, 'end_idx': 943000},
    ]
    
    print("=== UniMolV2 ===")
    
    # 
    print(f": {input_file}")
    df = pd.read_csv(input_file)
    
    # SMILES
    if 'SMILES' in df.columns:
        smiles_col = 'SMILES'
    elif 'smiles' in df.columns:
        smiles_col = 'smiles'
    else:
        smiles_col = df.columns[0]
    
    problematic_molecules = []
    cleaned_batches_data = []  # ：
    
    # 
    for batch_info in failed_batches:
        batch_idx = batch_info['batch_idx']
        start_idx = batch_info['start_idx']
        end_idx = batch_info['end_idx']
        
        print(f"\n {batch_idx} ( {start_idx}-{end_idx-1})")
        
        # 
        batch_df = df.iloc[start_idx:end_idx].copy()
        batch_smiles = batch_df[smiles_col].tolist()
        
        # （）
        batch_problematic_indices = []
        
        # 1: （）
        print(f"  ...")
        found_count = 0
        for i in tqdm(range(len(batch_smiles)), desc=f" {batch_idx}"):
            smiles = batch_smiles[i]
            is_valid, error_type, error_msg = test_molecule_unimolv2_direct(smiles)
            
            if not is_valid and error_type == "kekulize_error":
                global_idx = start_idx + i
                problematic_molecules.append({
                    'index': global_idx,
                    'batch_idx': batch_idx,
                    'smiles': smiles,
                    'error': error_msg
                })
                batch_problematic_indices.append(i)  # 
                found_count += 1
                if found_count <= 3:  # 3
                    print(f"    :  {global_idx}")
                    print(f"    SMILES: {smiles}")
                    print(f"    : {error_msg}")
        
        if found_count == 0:
            print(f"  ： {batch_idx} ")
        else:
            print(f"   {batch_idx}  {found_count} ")
        
        # （）
        if batch_problematic_indices:
            # 
            batch_df_clean = batch_df.drop(batch_df.index[batch_problematic_indices])
        else:
            batch_df_clean = batch_df
        
        # （）
        batch_df_clean['batch_idx'] = batch_idx
        
        # 
        cleaned_batches_data.append(batch_df_clean)
    
    if problematic_molecules:
        # 
        pd.DataFrame(problematic_molecules).to_csv(problematic_molecules_file, index=False)
        print(f"\n {len(problematic_molecules)} ")
        print(f": {problematic_molecules_file}")
        
        # 
        print("\n...")
        problematic_indices = [mol['index'] for mol in problematic_molecules]
        
        df_clean = df.drop(index=problematic_indices)
        df_clean.reset_index(drop=True, inplace=True)
        
        df_clean.to_csv(output_file, index=False)
        print(f": {output_file}")
        print(f": {len(df)}")
        print(f": {len(problematic_indices)}")
        print(f": {len(df_clean)}")
        
        # 
        if cleaned_batches_data:
            combined_cleaned_batches = pd.concat(cleaned_batches_data, ignore_index=True)
            # batch_idx，
            # combined_cleaned_batches = combined_cleaned_batches.drop('batch_idx', axis=1)
            combined_cleaned_batches.to_csv(cleaned_batches_file, index=False)
            print(f"\n: {cleaned_batches_file}")
            print(f": {len(combined_cleaned_batches)}")
        
        # 
        print("\n:")
        batch_stats = {}
        for mol in problematic_molecules:
            batch_idx = mol['batch_idx']
            batch_stats[batch_idx] = batch_stats.get(batch_idx, 0) + 1
        
        for batch_idx in sorted(batch_stats.keys()):
            print(f"   {batch_idx}: {batch_stats[batch_idx]} ")
        
        # 
        print("\n:")
        for batch_df_clean in cleaned_batches_data:
            batch_idx = batch_df_clean['batch_idx'].iloc[0]
            print(f"   {batch_idx}: {len(batch_df_clean)} molecules")
        
    else:
        print("\n！")
        # ，
        if cleaned_batches_data:
            combined_cleaned_batches = pd.concat(cleaned_batches_data, ignore_index=True)
            combined_cleaned_batches.to_csv(cleaned_batches_file, index=False)
            print(f"\n: {cleaned_batches_file}")
            print(f": {len(combined_cleaned_batches)}")

if __name__ == "__main__":
    main()